In [1]:
import base64
import csv
import io
import fitz  # PyMuPDF


def matrix_to_csv(data_matrix):
    """Converts a 2D matrix (list of lists) into a clean CSV string."""
    if not data_matrix:
        return ""
    output = io.StringIO()
    writer = csv.writer(output)
    for row in data_matrix:
        cleaned_row = [
            str(cell).replace("\n", " ").strip() if cell is not None else ""
            for cell in row
        ]
        writer.writerow(cleaned_row)
    return output.getvalue().strip()


def get_surrounding_text(text_blocks, bbox, v_margin=100, h_margin=50):
    """Finds text blocks near a visual element's bounding box (captions/context)."""
    context = []
    element_rect = fitz.Rect(bbox)

    for block in text_blocks:
        if block[6] == 0:  # Text block
            block_rect = fitz.Rect(block[:4])

            # Check both vertical and horizontal proximity
            v_close = (
                abs(element_rect.y0 - block_rect.y1) < v_margin
                or abs(block_rect.y0 - element_rect.y1) < v_margin
            )
            h_overlap = (
                block_rect.x1 >= element_rect.x0 - h_margin
                and block_rect.x0 <= element_rect.x1 + h_margin
            )

            if v_close and h_overlap:
                context.append(block[4].strip())

    return " ".join(context)


def bytes_to_base64_url(image_bytes: bytes, ext: str = "png") -> str:
    """Formats raw bytes into a Base64 Data URL for Vision LLM APIs."""
    mime = "jpeg" if ext.lower() in ["jpg", "jpeg"] else ext.lower()
    b64 = base64.b64encode(image_bytes).decode("utf-8")
    return f"data:image/{mime};base64,{b64}"


def pdf_loader(file_path):
    """Parses PDF into Prose Text (no tables), CSV Tables, and Vision-ready Images/Charts with Context."""
    doc = fitz.open(file_path)

    pdf_content = {
        "texts": [],
        "tables": [],
        "images": [],
        "charts": [],
    }

    for page in doc:
        text_blocks = page.get_text("blocks")
        tables = page.find_tables()
        table_rects = [fitz.Rect(tab.bbox) for tab in tables]

        # 1. Prose Text Only (Filters out text inside tables)
        page_prose = []
        for block in text_blocks:
            if block[6] == 0:
                block_rect = fitz.Rect(block[:4])
                if not any(
                    block_rect.intersects(t_rect) for t_rect in table_rects
                ):
                    page_prose.append(block[4].strip())

        pdf_content["texts"].append("\n\n".join(page_prose))

        # 2. Tables (CSV + Context)
        for tab in tables:
            raw_matrix = tab.extract()
            context = get_surrounding_text(text_blocks, tab.bbox)
            pdf_content["tables"].append({
                "csv": matrix_to_csv(raw_matrix),
                "context": context,
            })

        # 3. Images (Page-specific extraction)
        for img_info in page.get_images(full=True):
            xref = img_info[0]
            base_img = doc.extract_image(xref)
            rects = page.get_image_rects(xref)

            if rects:
                # Get context from the first placement on the page
                context = get_surrounding_text(text_blocks, rects[0])
                b64_url = bytes_to_base64_url(
                    base_img["image"], base_img["ext"]
                )

                pdf_content["images"].append({
                    "context": context,
                    "image_b64": b64_url,
                    "raw_bytes": base_img["image"],
                    "ext": base_img["ext"],
                })

        # 4. Charts / Vector Graphics (Deduplicated & Boundary-Safe)
        processed_chart_rects = []
        for draw in page.get_drawings():
            rect = fitz.Rect(draw["rect"]) & page.rect  # Clip to page boundary

            # Filter size and check if vector drawing sits inside a table
            if rect.width > 150 and rect.height > 100:
                if any(rect.intersects(t_rect) for t_rect in table_rects):
                    continue

                # Deduplicate: Skip if this rect heavily overlaps with an already extracted chart
                if any(rect.intersects(p_rect) for p_rect in processed_chart_rects):
                    continue

                processed_chart_rects.append(rect)

                pix = page.get_pixmap(clip=rect)
                chart_bytes = pix.tobytes("png")
                context = get_surrounding_text(text_blocks, rect)

                pdf_content["charts"].append({
                    "context": context,
                    "image_b64": bytes_to_base64_url(chart_bytes, "png"),
                    "raw_bytes": chart_bytes,
                    "ext": "png",
                })

    doc.close()
    return pdf_content


# Usage
file_path = "./data.pdf"
pdf_content = pdf_loader(file_path)

print(f"{len(pdf_content['texts'])} text pages parsed")
print(f"{len(pdf_content['images'])} images found")
print(f"{len(pdf_content['charts'])} vector charts found")
print(f"{len(pdf_content['tables'])} tables found")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
4 text pages parsed
3 images found
0 vector charts found
2 tables found


In [2]:
for table in pdf_content['tables']:
    print(f"Table: {table}")

for text in pdf_content['texts']:
    print(f"Text: {text}")

Table: {'csv': 'Model,Year,Parameters,Layers,Primary Use Case\r\nBERT-Base,2018,110M,12,Bidirectional language understanding\r\nGPT-2,2019,1.5B,48,Autoregressive text generation\r\nT5-Large,2019,770M,24,Text-to-text transfer learning\r\nGPT-3,2020,175B,96,Few-shot general-purpose generation\r\nPaLM,2022,540B,118,Large-scale reasoning and generation', 'context': '2. Model Comparison Transformer-based models vary widely in scale and intended use. The table below summarizes a few\nwidely referenced models. Model\nYear\nParameters\nLayers\nPrimary Use Case BERT-Base\n2018\n110M\n12\nBidirectional language understanding GPT-2\n2019\n1.5B\n48\nAutoregressive text generation T5-Large\n2019\n770M\n24\nText-to-text transfer learning GPT-3\n2020\n175B\n96\nFew-shot general-purpose generation PaLM\n2022\n540B\n118\nLarge-scale reasoning and generation Table 1. Approximate scale of selected Transformer models.'}
Table: {'csv': 'Hyperparameter,Typical Range\r\nNumber of layers,6 - 96+\r\nAttention 

In [5]:
import base64
import io
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from PIL import Image
from tenacity import (
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
)

load_dotenv(override=True)

# 2. Initialize Multimodal LLM
llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0.1,
    max_tokens=4096,  # Give enough head-room for full image description
    reasoning_effort="none",  # Completely disables internal reasoning tokens
)


# 3. Image Compression Helper
def compress_bytes_to_b64(raw_bytes: bytes, max_dim: int = 768) -> str:
    """Downscales image to 768px max dimension to keep token usage low."""
    img = Image.open(io.BytesIO(raw_bytes))
    if img.mode in ("RGBA", "P"):
        img = img.convert("RGB")
    img.thumbnail((max_dim, max_dim), Image.Resampling.LANCZOS)

    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=75)
    b64_str = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64_str}"


# 4. LLM Invocation with Exponential Backoff Retry (Handles RateLimitError / 429)
@retry(
    stop=stop_after_attempt(5),  # Retry up to 5 times
    wait=wait_exponential(multiplier=2, min=4, max=20),  # Wait 4s, 8s, 16s...
    retry=retry_if_exception_type(Exception),
    reraise=True,
)
def invoke_llm_with_retry(llm: ChatGroq, message: HumanMessage):
    return llm.invoke([message])


# 5. Worker Task Function for ThreadPoolExecutor
def process_single_visual(args: tuple) -> dict:
    idx, visual_item = args
    surrounding_text = visual_item.get("context", "None provided.")

    prompt = f"""
Task: Analyze the image and surrounding document text as a Vision-Language Model for a RAG system.

Surrounding text:
{surrounding_text}

Generate ONE self-contained, retrieval-friendly description that enables an LLM to answer questions about the image without seeing it.

Rules:

Start exactly with: "This image contains..."
Identify the visual type and purpose.
Extract key visible information: titles, captions, labels, legends, headers, values, units, dates, names, equations, and technical terms.
For charts/graphs, describe measurements, comparisons, exact or approximate values, highest/lowest items, ordering, trends, and important relationships.
For tables, describe purpose, headers, relevant rows, values, comparisons, and patterns.
Use surrounding text to explain context, meaning, and relevance.
Include important conclusions supported by the visual.
Preserve useful technical terminology and synonyms for retrieval.
Prioritize factual accuracy and semantic information over decorative details.
Mention colors only when they identify meaningful data or categories.
Never invent or guess information. Mark unclear values as approximate.
Avoid unnecessary repetition and verbose visual descriptions.

OUTPUT CONSTRAINT:

Output ONLY the final description.
The description MUST be between 250 and 350 tokens.
Aim for approximately 300 tokens.
""".strip()

    if "raw_bytes" in visual_item:
        image_url_str = compress_bytes_to_b64(visual_item["raw_bytes"])
    elif "image_b64" in visual_item:
        b64_val = visual_item["image_b64"]
        image_url_str = (
            b64_val
            if b64_val.startswith("data:image/")
            else f"data:image/png;base64,{b64_val}"
        )
    else:
        raise ValueError("Missing raw_bytes or image_b64 in visual item.")

    message = HumanMessage(
        content=[
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": image_url_str}},
        ]
    )

    # Executing through tenacity retry wrapper
    response = invoke_llm_with_retry(llm, message)
    return {"index": idx, "description": response.content.strip()}


visual_items = pdf_content.get("images", []) + pdf_content.get("charts", [])

if visual_items:
    # Set max_workers=2 or 3 to prevent overwhelming the free tier rate limit
    MAX_CONCURRENT_WORKERS = min(3, len(visual_items))

    print(
        f"Found {len(visual_items)} visual items. Processing in parallel using ThreadPoolExecutor (workers={MAX_CONCURRENT_WORKERS})...\n"
    )

    rag_chunks = []
    indexed_items = list(enumerate(visual_items, start=1))

    with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_WORKERS) as executor:
        # Submit tasks
        future_to_item = {
            executor.submit(process_single_visual, item): item[0]
            for item in indexed_items
        }

        # Collect results as they complete
        for future in as_completed(future_to_item):
            item_idx = future_to_item[future]
            try:
                result = future.result()
                rag_chunks.append(result)
                print(f"Successfully generated chunk for visual {item_idx}.")
            except Exception as exc:
                print(
                    f"Visual item {item_idx} generated an exception: {exc}"
                )

    rag_chunks.sort(key=lambda x: x["index"])

    print("Output")
    for chunk in rag_chunks:
        print(f"{chunk['index']}")
        print(chunk["description"])
        print("end")
else:
    print("No visual elements found in the document.")


Found 3 visual items. Processing in parallel using ThreadPoolExecutor (workers=3)...

Successfully generated chunk for visual 2.
Successfully generated chunk for visual 1.
Successfully generated chunk for visual 3.
Output
1
This image contains a schematic flowchart illustrating the core architecture of a standard Transformer encoder block, labeled as "Figure 1. Simplified structure of a Transformer encoder block." The diagram visualizes the vertical data flow from bottom to top, starting with a yellow box labeled "Input Embeddings + Positional Encoding." This input feeds into a large gray container labeled "Transformer Encoder Block (x N)," indicating that this specific sequence of operations is stacked N times in the model. Inside this block, the first processing layer is a blue box labeled "Multi-Head Self-Attention." This is immediately followed by a gray box labeled "Add & Layer Norm," representing a residual connection and normalization step. The subsequent layer is a green box la